In [ ]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
import pandas as pd,os,json

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5-20251001"


df = pd.read_csv(r"telehealth.csv")

def get_df_info():
    temp_df = df.copy()
    temp_df.fillna("N/A", inplace=True)

    return {
        "columns": temp_df.columns.tolist(),
        "rows": len(temp_df),
        "data_types": temp_df.dtypes.astype(str).to_dict(),
        "top_5_rows": temp_df.head().to_dict(orient="records"),
    }

def run_cohort_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        
        total_patients=len(group)
        active_patients=(
            group["retention_status"].astype(str).str.lower().isin(["active", "completed"]).sum()
        )
        
        churned_patients=total_patients-active_patients
        
        retention_rate=round((active_patients/total_patients)*100,2)
        
        result[program]={
            "number_of_patients":total_patients,
            "average_treatment_duration_weeks":round(group["treatment_duration_weeks"].mean(),2),
            "active_patients":int(active_patients),
            "churned_patients":int(churned_patients),
            "retention_rate_percent":retention_rate,
        }

    return result

def run_outcome_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        retention_rate = round(
            (group["retention_status"].astype(str).str.lower().isin(["active", "completed"]).sum() / len(group)) * 100, 2
        )
        duration_q1 = round(group["treatment_duration_weeks"].quantile(0.25), 2)
        duration_median = round(group["treatment_duration_weeks"].quantile(0.5), 2)
        duration_q3 = round(group["treatment_duration_weeks"].quantile(0.75), 2)
        duration_mean = round(group["treatment_duration_weeks"].mean(), 2)
        
        churned = group[group["retention_status"].astype(str).str.lower().eq("dropped off")]
        
        early_dropoff = len(churned[churned["treatment_duration_weeks"] <= duration_q1])
        mid_dropoff = len(churned[(churned["treatment_duration_weeks"] > duration_q1) & 
                                (churned["treatment_duration_weeks"] <= duration_q3)])
        late_dropoff = len(churned[churned["treatment_duration_weeks"] > duration_q3])
        
        result[program] = {
            "retention_rate_percent": retention_rate,
            "duration_trends": {
                "mean_weeks": duration_mean,
                "median_weeks": duration_median,
                "q1_weeks": duration_q1,
                "q3_weeks": duration_q3,
            },
            "drop_off_points": {
                "early_stage_patients": int(early_dropoff),
                "mid_stage_patients": int(mid_dropoff),
                "late_stage_patients": int(late_dropoff),
            }
        }
    
    return result

def flag_anomalies(df):
    result = {}
    
    for program, group in df.groupby("program_type"):
        q1 = group["treatment_duration_weeks"].quantile(0.25)
        q3 = group["treatment_duration_weeks"].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        
        short_duration_outliers = group[group["treatment_duration_weeks"] < lower_bound]
        short_duration_count = len(short_duration_outliers)
        
        churned = group[group["retention_status"].astype(str).str.lower().eq("dropped off")]
        churn_rate = round((len(churned) / len(group)) * 100, 2)
        
        abnormal_churn = churn_rate > 25.0
        
        result[program] = {
            "short_duration_outliers": {
                "count": int(short_duration_count),
                "threshold_weeks": round(lower_bound, 2),
                "affected_patients_percent": round((short_duration_count / len(group)) * 100, 2),
                "flagged": short_duration_count > 0,
            },
            "abnormal_drop_off_clusters": {
                "churn_rate_percent": churn_rate,
                "flagged": abnormal_churn,
                "interpretation": "High churn detected" if abnormal_churn else "Normal churn levels"
            }
        }
    
    return result


tools = [
    {
        "name": "get_df_info",
        "description": "Returns raw dataset structure: column names, data types, row count, and sample rows. Use this only to inspect what data exists before running any analysis. Provides no statistics, comparisons, trends, or risk flags.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_cohort_analysis",
        "description": "Returns a point-in-time snapshot per program for side-by-side comparison: patient counts and retention percentage. Use for ranking or comparing programs against each other, including ranking by churn or retention. Does not analyze drop-off timing or flag statistical risk — for timing use run_outcome_analysis, for risk thresholds use flag_anomalies.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_outcome_analysis",
        "description": "Returns time-based behavior per program: how treatment length is distributed, and which stage (early, mid, late) churned patients left at. Use for understanding when or at what point in treatment patients drop off. Does not rank programs against each other or flag statistical risk — for ranking use run_cohort_analysis, for risk thresholds use flag_anomalies.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "flag_anomalies",
        "description": "Returns statistical risk flags per program: treatment durations that are extreme outliers (IQR method) and churn rates exceeding a 25% threshold. Use for risk detection or exception flagging, not raw counts or timing. Does not provide patient counts or drop-off timing — for counts use run_cohort_analysis, for timing use run_outcome_analysis.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

def add_user_message(messages, message):
    messages.append(
        {
            "role": "user",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def add_assistant_message(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def chat(messages, system=None, temperature=0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "tools": tools,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    return client.messages.create(**params)


def text_from_message(message):
    texts = []
    for block in message.content:
        if block.type == "text":
            texts.append(block.text)

    return "\n".join(texts)


def run_tool(tool_name, tool_input):
    if tool_name == "get_df_info":
        return get_df_info()
    elif tool_name =='run_cohort_analysis':
        return run_cohort_analysis(df)
    elif tool_name == 'run_outcome_analysis':
        return run_outcome_analysis(df)
    elif tool_name == 'flag_anomalies':
        return flag_anomalies(df)

    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_results = []

    for block in message.content:

        if block.type != "tool_use":
            continue

        try:
            result = run_tool(block.name, block.input)

            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

        except Exception as e:
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(e),
                    "is_error": True,
                }
            )

    return tool_results


def run_conversation(messages):
    while True:
        response = chat(messages)
        
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_requests = [block for block in response.content if block.type == "tool_use"]
        for tool_request in tool_requests:
            pass  # tracking optional

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages


messages = []
add_user_message(
    messages,
    "Analyze the telehealth patient dataset and provide insights on patient retention and treatment outcomes across different programs. Highlight any anomalies or outliers in the data."
)

messages = run_conversation(messages)

I'll analyze the telehealth patient dataset comprehensively. Let me start by examining the data structure, then run analyses on retention, outcomes, and anomalies.
## Comprehensive Telehealth Patient Dataset Analysis

### **Dataset Overview**
- **Total Patients:** 300
- **Programs:** 3 (Peptides, Testosterone, Weight Loss)
- **Key Metrics:** Patient counts, treatment duration, retention status, and drop-off reasons

---

### **Patient Retention Summary**

| Program | Patients | Active | Churned | Retention Rate |
|---------|----------|--------|---------|-----------------|
| **Weight Loss** | 97 | 75 | 22 | **77.32%** ✅ |
| **Peptides** | 91 | 67 | 24 | **73.63%** |
| **Testosterone** | 112 | 80 | 32 | **71.43%** |

**Key Finding:** Weight Loss program shows the strongest retention, while Testosterone has the highest absolute churn (32 patients).

---

### **Treatment Duration & Outcomes**

#### **Peptides Program**
- **Average Duration:** 10.31 weeks (shortest)
- **Median Duration:** 1